# Weighted-Average TE and Lag vs CAMELS Attributes

## What we are doing
Last week we used **max TE** and **peak lag** per forcing per basin, then correlated those with 52 CAMELS attributes. The scatter plots were noisy and we could not pick a clean attribute story.

This week we replace max with a **weighted average** over significant lags.

For each basin and each forcing (PRCP, SRAD, Tair, VP) we compute:

- **τ\*** = Σ TE(τ)·τ  /  Σ TE(τ),  τ in {1..30}, only significant lags
- **TE\*** = Σ TE(τ)·τ  /  Σ τ,  τ in {1..30}, only significant lags

Then we do the same kind of analysis as last week:

1. One CSV with τ\* and TE\* per basin per forcing (671 basins × 4 forcings).
2. Scatter plots: each forcing's τ\* vs each of the 52 attributes. Same for TE\*.
3. Two CONUS maps:
   - Map A: τ\* with 4 subplots (one per forcing)
   - Map B: TE\* with 4 subplots (one per forcing)
4. Compare with last week's max-TE / peak-lag results and discuss.

## Inputs
- Raw TE values and shuffle thresholds from the older folder:
  `ALL CAMELS BASIN (FORCINGS TO STREAMFLOW)\outputs\summary_csv`
- 52 CAMELS attributes from the main CAMELS Research folder

## Outputs (this project folder)
`Weighted_Average_TE vs Attributes\outputs\`
- `summary_csv\` — basin-level τ\* and TE\* table, correlation tables
- `pdf\` — scatter plots and CONUS maps
- `logs\` — run logs

## Note on the formula
Both τ\* and TE\* are TE-weighted, so they lean toward the lags where TE is largest. If a basin's TE is strongest at short lags, both τ\* and TE\* will reflect that. We will compare this with last week's max-based results and decide which view is more useful.

## Project setup

Set folder paths and create the output folders. Nothing is computed yet. We just check that the input files we need from last week are reachable.

In [24]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from matplotlib.backends.backend_pdf import PdfPages
from scipy import stats
import os
import pandas as pd
from pathlib import Path
import matplotlib.patches as mpatches
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.io.shapereader import natural_earth, Reader

In [2]:
# This project folder (where we save outputs)
project_folder = Path(r"C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Research -CAMELS\Weighted_Average_TE vs Attributes")

# Old TE project folder (where we read raw TE and shuffle results from)
te_folder = Path(r"C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Python\Jupyter Notebook\CAMELS Research\Final research code\ALL CAMELS BASIN (FORCINGS TO STREAMFLOW)")

# Output subfolders
out_csv  = project_folder / "outputs" / "summary_csv"
out_pdf  = project_folder / "outputs" / "pdf"
out_logs = project_folder / "outputs" / "logs"

for folder in [out_csv, out_pdf, out_logs]:
    folder.mkdir(parents=True, exist_ok=True)

# Where TE results live in the old folder
te_csv_folder = te_folder / "outputs" / "summary_csv"

print("Project folder exists:", project_folder.exists())
print("TE folder exists:     ", te_folder.exists())
print("TE summary_csv exists:", te_csv_folder.exists())
print()
print("Output folders ready:")
print(" ", out_csv)
print(" ", out_pdf)
print(" ", out_logs)

Project folder exists: True
TE folder exists:      True
TE summary_csv exists: True

Output folders ready:
  C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Research -CAMELS\Weighted_Average_TE vs Attributes\outputs\summary_csv
  C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Research -CAMELS\Weighted_Average_TE vs Attributes\outputs\pdf
  C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Research -CAMELS\Weighted_Average_TE vs Attributes\outputs\logs


## Load the TE shuffle file

Read the same TE shuffle file we used last week. It has TE values and shuffle test results for 671 basins, 4 forcings, lags 1 to 30. We will use the `sig` column to keep only significant TE values when we compute τ* and TE* in the next step.

In [3]:
# TE shuffle file from the old folder
te_path = te_csv_folder / "TE_shuffle_all671_tau1_30_PRCPspecial_bins5.csv"

te_df = pd.read_csv(te_path, dtype={"gauge_id": str})
te_df["gauge_id"] = te_df["gauge_id"].str.zfill(8)

print("Shape:", te_df.shape)
print()
print("Columns:", te_df.columns.tolist())
print()
print("First 5 rows:")
print(te_df.head())
print()
print("Unique forcings:", te_df["Source"].unique())
print("Unique lags:    ", sorted(te_df["Lag"].unique()))
print()
print("Significant rows (sig=True):", (te_df["sig"] == True).sum())
print("Non-significant rows       :", (te_df["sig"] == False).sum())

Shape: (80520, 10)

Columns: ['gauge_id', 'huc_02', 'Source', 'Target', 'Bins', 'Lag', 'TE_obs', 'thr95', 'pval', 'sig']

First 5 rows:
   gauge_id  huc_02 Source Target  Bins  Lag    TE_obs     thr95      pval  \
0  01013500       1   PRCP      Q     5    1  0.002761  0.001187  0.004975   
1  01013500       1   PRCP      Q     5    2  0.001754  0.001171  0.004975   
2  01013500       1   PRCP      Q     5    3  0.001439  0.001204  0.009950   
3  01013500       1   PRCP      Q     5    4  0.000913  0.001174  0.258706   
4  01013500       1   PRCP      Q     5    5  0.001240  0.001064  0.024876   

     sig  
0   True  
1   True  
2   True  
3  False  
4   True  

Unique forcings: ['PRCP' 'SRAD' 'Tair' 'VP']
Unique lags:     [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64

The file has 80,520 rows = 671 basins × 4 forcings × 30 lags. Significant TE rows: 34,511. Non-significant: 46,009. About 43% of TE values pass the shuffle test. Forcings and lags are as expected. This matches last week's notebook, so the input is correct.

## Compute τ* and TE* per basin per forcing

For each basin and each forcing we use only significant TE rows. Then:

- τ* = sum of (TE_obs × Lag) over significant lags, divided by sum of TE_obs over significant lags.
- TE* = sum of (TE_obs × Lag) over significant lags, divided by sum of Lag over significant lags.

If a basin has zero significant lags for a forcing, both τ* and TE* are set to NaN. We also save the number of significant lags per basin per forcing as `n_sig`, so we know how trustworthy the value is.

In [4]:
# keep only significant rows
te_sig = te_df[te_df["sig"] == True].copy()

# for each basin + forcing, compute the two weighted averages
def weighted_metrics(group):
    te  = group["TE_obs"].values
    lag = group["Lag"].values
    sum_te_lag = (te * lag).sum()
    sum_te     = te.sum()
    sum_lag    = lag.sum()
    tau_star = sum_te_lag / sum_te  if sum_te  > 0 else float("nan")
    te_star  = sum_te_lag / sum_lag if sum_lag > 0 else float("nan")
    return pd.Series({
        "tau_star": tau_star,
        "TE_star":  te_star,
        "n_sig":    len(group),
    })

weighted_df = te_sig.groupby(["gauge_id", "Source"]).apply(weighted_metrics).reset_index()

print("Long format shape:", weighted_df.shape)
print()
print("First 12 rows:")
print(weighted_df.head(12))
print()
print("Basin count per forcing (basins with at least 1 significant lag):")
print(weighted_df.groupby("Source")["gauge_id"].count())
print()
print("Summary stats:")
print(weighted_df.groupby("Source")[["tau_star", "TE_star", "n_sig"]].describe().round(4))

Long format shape: (2493, 5)

First 12 rows:
    gauge_id Source   tau_star   TE_star  n_sig
0   01013500   PRCP   8.052347  0.001382    8.0
1   01013500   SRAD  14.034215  0.002189   26.0
2   01013500   Tair  15.530727  0.003327   30.0
3   01013500     VP  15.950455  0.002133   25.0
4   01022500   PRCP   6.600209  0.002255    9.0
5   01022500   SRAD   6.049889  0.003144    8.0
6   01022500   Tair  14.907453  0.004956   30.0
7   01022500     VP  15.187192  0.004334   30.0
8   01030500   PRCP   5.117754  0.001892   13.0
9   01030500   SRAD   8.958003  0.002623   13.0
10  01030500   Tair  14.064139  0.003848   30.0
11  01030500     VP  14.168115  0.002906   29.0

Basin count per forcing (basins with at least 1 significant lag):
Source
PRCP    670
SRAD    653
Tair    593
VP      577
Name: gauge_id, dtype: int64

Summary stats:
       tau_star                                                         \
          count     mean     std  min      25%      50%      75%   max   
Source          

C:\Users\ppaudel2\AppData\Local\Temp\ipykernel_7712\3047140945.py:19: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  weighted_df = te_sig.groupby(["gauge_id", "Source"]).apply(weighted_metrics).reset_index()


2,493 basin-forcing pairs. Basin counts match last week: PRCP 670, SRAD 653, Tair 593, VP 577.

Early patterns from the summary stats:

- **τ\* mean by forcing:** PRCP ≈ 8.6 days, SRAD ≈ 12.4, VP ≈ 14.0, Tair ≈ 15.4. So rainfall information transfers to streamflow at shorter lags on average, while temperature information transfers at longer lags. This is consistent with the idea that PRCP drives quick runoff response, while Tair acts more through slower processes like snowmelt and ET seasonality.

- **TE\* mean by forcing:** SRAD ≈ 0.0026 and Tair ≈ 0.0025 are highest, PRCP ≈ 0.0015 lowest. This is partly because of the τ-weighting: forcings with significant TE at longer lags get a larger TE\*. We will check this against the τ\* values later.

- **n_sig (number of significant lags) by forcing:** Tair has on average 19 significant lags out of 30, PRCP only about 10. So Tair gives sustained information over many lags, while PRCP information is more concentrated.

- Both τ\* and TE\* have min = 1.0 in some cases. That means at least one basin has only one significant lag at τ = 1. With only one point, τ\* = 1 by definition. We should keep an eye on basins with very low n_sig — those values are not really an "average".

## Pivot to wide format — one row per basin

Reshape the long table so each basin has one row with 8 numbers: τ* and TE* for each of the 4 forcings. This wide format is what we will merge with the 52 attributes in the next step.

In [5]:
# pivot τ* and TE* separately, then merge
tau_wide = weighted_df.pivot(index="gauge_id", columns="Source", values="tau_star")
tau_wide.columns = [f"tau_star_{c}" for c in tau_wide.columns]

te_wide = weighted_df.pivot(index="gauge_id", columns="Source", values="TE_star")
te_wide.columns = [f"TE_star_{c}" for c in te_wide.columns]

nsig_wide = weighted_df.pivot(index="gauge_id", columns="Source", values="n_sig")
nsig_wide.columns = [f"n_sig_{c}" for c in nsig_wide.columns]

# merge into one wide DataFrame
wide_df = tau_wide.join(te_wide).join(nsig_wide).reset_index()

print("Wide format shape:", wide_df.shape)
print()
print("Columns:", wide_df.columns.tolist())
print()
print("First 3 rows:")
print(wide_df.head(3))
print()
print("Missing values per column:")
print(wide_df.isna().sum())

# save the wide-format table
out_wide_path = out_csv / "weighted_TE_tau_per_basin.csv"
wide_df.to_csv(out_wide_path, index=False)
print()
print("Saved:", out_wide_path)

Wide format shape: (671, 13)

Columns: ['gauge_id', 'tau_star_PRCP', 'tau_star_SRAD', 'tau_star_Tair', 'tau_star_VP', 'TE_star_PRCP', 'TE_star_SRAD', 'TE_star_Tair', 'TE_star_VP', 'n_sig_PRCP', 'n_sig_SRAD', 'n_sig_Tair', 'n_sig_VP']

First 3 rows:
   gauge_id  tau_star_PRCP  tau_star_SRAD  tau_star_Tair  tau_star_VP  \
0  01013500       8.052347      14.034215      15.530727    15.950455   
1  01022500       6.600209       6.049889      14.907453    15.187192   
2  01030500       5.117754       8.958003      14.064139    14.168115   

   TE_star_PRCP  TE_star_SRAD  TE_star_Tair  TE_star_VP  n_sig_PRCP  \
0      0.001382      0.002189      0.003327    0.002133         8.0   
1      0.002255      0.003144      0.004956    0.004334         9.0   
2      0.001892      0.002623      0.003848    0.002906        13.0   

   n_sig_SRAD  n_sig_Tair  n_sig_VP  
0        26.0        30.0      25.0  
1         8.0        30.0      30.0  
2        13.0        30.0      29.0  

Missing values per c

671 basins, one row each. 8 weighted metrics per basin (4 τ* and 4 TE*) plus 4 n_sig columns. Missing-value pattern matches last week: 1 basin has no significant PRCP TE, 18 have no significant SRAD TE, 78 have no significant Tair TE, 94 have no significant VP TE. These are the same basins that were missing peak TE / peak lag last week, so the comparison stays consistent. File saved to `outputs\summary_csv\weighted_TE_tau_per_basin.csv`.

## Load the 52 attributes and merge with τ* and TE*

Read the same attribute CSV used last week (`camels_attributes_combined_671basins.csv`). Merge it with the wide-format τ* and TE* table on `gauge_id`. After this we have one row per basin with τ*, TE*, n_sig, and all 52 attributes side by side.

In [6]:
# attribute CSV path
attr_path = r"C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Python\Jupyter Notebook\CAMELS Research\Final research code\TE basin to basin analysis with 60 attributes\outputs\summary_csv\camels_attributes_combined_671basins.csv"

attr_df = pd.read_csv(attr_path, dtype={"gauge_id": str})
attr_df["gauge_id"] = attr_df["gauge_id"].str.zfill(8)

print("Attributes shape:", attr_df.shape)
print()
print("First 5 attribute columns:", attr_df.columns.tolist()[:5])
print("Last 5 attribute columns: ", attr_df.columns.tolist()[-5:])
print()

# merge with wide-format τ* and TE* table
merged_df = wide_df.merge(attr_df, on="gauge_id", how="left")

print("Merged shape:", merged_df.shape)
print()

# save merged file
out_merged_path = out_csv / "weighted_TE_tau_with_attributes.csv"
merged_df.to_csv(out_merged_path, index=False)
print("Saved:", out_merged_path)

Attributes shape: (671, 60)

First 5 attribute columns: ['gauge_id', 'huc_02', 'gauge_name', 'p_mean', 'pet_mean']
Last 5 attribute columns:  ['gvf_diff', 'dom_land_cover_frac', 'dom_land_cover', 'root_depth_50', 'root_depth_99']

Merged shape: (671, 72)

Saved: C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Research -CAMELS\Weighted_Average_TE vs Attributes\outputs\summary_csv\weighted_TE_tau_with_attributes.csv


## Confirm the attribute list

Print all 60 columns in the attribute file. Last week we used 52 attributes for correlation. We need to confirm which 52 — so the new analysis matches last week exactly.

In [7]:
# print all attribute columns
print("All 60 columns in attribute file:")
for i, col in enumerate(attr_df.columns.tolist(), 1):
    print(f"  {i:2}. {col}")

All 60 columns in attribute file:
   1. gauge_id
   2. huc_02
   3. gauge_name
   4. p_mean
   5. pet_mean
   6. p_seasonality
   7. frac_snow
   8. aridity
   9. high_prec_freq
  10. high_prec_dur
  11. high_prec_timing
  12. low_prec_freq
  13. low_prec_dur
  14. low_prec_timing
  15. geol_1st_class
  16. glim_1st_class_frac
  17. geol_2nd_class
  18. glim_2nd_class_frac
  19. carbonate_rocks_frac
  20. geol_porostiy
  21. geol_permeability
  22. q_mean
  23. runoff_ratio
  24. slope_fdc
  25. baseflow_index
  26. stream_elas
  27. q5
  28. q95
  29. high_q_freq
  30. high_q_dur
  31. low_q_freq
  32. low_q_dur
  33. zero_q_freq
  34. hfd_mean
  35. soil_depth_pelletier
  36. soil_depth_statsgo
  37. soil_porosity
  38. soil_conductivity
  39. max_water_content
  40. sand_frac
  41. silt_frac
  42. clay_frac
  43. water_frac
  44. organic_frac
  45. other_frac
  46. gauge_lat
  47. gauge_lon
  48. elev_mean
  49. slope_mean
  50. area_gages2
  51. area_geospa_fabric
  52. frac_forest

## Lock the final 50-attribute list

Last week's correlation file used 50 attributes (geol_1st_class and geol_2nd_class were excluded because they are categorical). We use the same 50 here so the comparison will be same.

In [10]:
# update exclude list to drop the 2 text geology columns too
exclude_cols = [
    "gauge_id", "huc_02", "gauge_name", "dom_land_cover",
    "gauge_lat", "gauge_lon",
    "high_prec_timing", "low_prec_timing",
    "geol_1st_class", "geol_2nd_class",
]

# final attribute list
attr_list = [c for c in attr_df.columns if c not in exclude_cols]

print("Final number of attributes:", len(attr_list))
print()

# verify all numeric
non_numeric = attr_df[attr_list].dtypes[
    ~attr_df[attr_list].dtypes.apply(lambda x: pd.api.types.is_numeric_dtype(x))
]
print("Non-numeric attributes:", "None" if len(non_numeric) == 0 else list(non_numeric.index))
print()
print("Matches last week's 50 attributes?", set(attr_list) == set(unique_attrs))

Final number of attributes: 50

Non-numeric attributes: None

Matches last week's 50 attributes? True


Final attribute list: 50 numeric CAMELS attributes. Same set as last week. Excluded: gauge_id, huc_02, gauge_name, dom_land_cover, gauge_lat, gauge_lon, high_prec_timing, low_prec_timing, geol_1st_class, geol_2nd_class.

## Correlate τ* and TE* with the 50 attributes (Pearson)

For each forcing (PRCP, SRAD, Tair, VP) and each of the 50 attributes, compute Pearson r for τ* vs attribute and for TE* vs attribute. NaN values are dropped pairwise. We also save the sample size n for each pair.

In [12]:
forcings = ["PRCP", "SRAD", "Tair", "VP"]

rows = []
for f in forcings:
    tau_col = f"tau_star_{f}"
    te_col  = f"TE_star_{f}"
    for attr in attr_list:
        # τ* vs attribute
        sub_tau = merged_df[[tau_col, attr]].dropna()
        n_tau = len(sub_tau)
        if n_tau >= 3:
            r_tau, _ = stats.pearsonr(sub_tau[tau_col], sub_tau[attr])
        else:
            r_tau = float("nan")

        # TE* vs attribute
        sub_te = merged_df[[te_col, attr]].dropna()
        n_te = len(sub_te)
        if n_te >= 3:
            r_te, _ = stats.pearsonr(sub_te[te_col], sub_te[attr])
        else:
            r_te = float("nan")

        rows.append({
            "forcing":   f,
            "attribute": attr,
            "n_tau":     n_tau,
            "r_tau":     r_tau,
            "n_TE":      n_te,
            "r_TE":      r_te,
        })

corr_df = pd.DataFrame(rows)

# save
out_corr_path = out_csv / "correlation_summary_weighted_TE_tau.csv"
corr_df.to_csv(out_corr_path, index=False)

print("Correlation table shape:", corr_df.shape)
print()
print("First 6 rows:")
print(corr_df.head(6).round(4))
print()
print("Sample size n per forcing (basins used in each correlation):")
print(corr_df.groupby("forcing")[["n_tau", "n_TE"]].first())
print()
print("Saved:", out_corr_path)

Correlation table shape: (200, 6)

First 6 rows:
  forcing       attribute  n_tau   r_tau  n_TE    r_TE
0    PRCP          p_mean    670 -0.1127   670  0.3155
1    PRCP        pet_mean    670  0.1013   670 -0.2049
2    PRCP   p_seasonality    670 -0.2555   670 -0.2895
3    PRCP       frac_snow    670  0.3054   670  0.1563
4    PRCP         aridity    670  0.2594   670 -0.2498
5    PRCP  high_prec_freq    670  0.0084   670 -0.3631

Sample size n per forcing (basins used in each correlation):
         n_tau  n_TE
forcing             
PRCP       670   670
SRAD       653   653
Tair       593   593
VP         577   577

Saved: C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Research -CAMELS\Weighted_Average_TE vs Attributes\outputs\summary_csv\correlation_summary_weighted_TE_tau.csv


In [13]:
# show the data that produced r = -0.1127 for PRCP τ* vs p_mean
check = merged_df[["gauge_id", "tau_star_PRCP", "p_mean"]].dropna()
print("Basins used:", len(check))
print()
print(check.head(10))
print()
# recompute r to confirm
r, _ = stats.pearsonr(check["tau_star_PRCP"], check["p_mean"])
print(f"Recomputed r = {r:.4f}")

Basins used: 670

   gauge_id  tau_star_PRCP    p_mean
0  01013500       8.052347  3.126679
1  01022500       6.600209  3.608126
2  01030500       5.117754  3.274405
3  01031500       4.833022  3.522957
4  01047000       4.256450  3.323146
5  01052500       1.452078  3.730858
6  01054200       3.785712  4.067132
7  01055000       3.721523  3.494183
8  01057000       4.373906  3.570500
9  01073000       8.434435  3.503025

Recomputed r = -0.1127


## Plot TE* vs each attribute, colored by τ* group

Create one PDF with 52 pages: 50 numeric scatter plots + 2 categorical boxplots (high_prec_timing, low_prec_timing). Each page has 4 subplots — one per forcing. Points are colored by τ* group using the same lag bins as last week.

In [16]:
# τ* color groups (same lag bins as last week)
def get_tau_group(tau):
    if tau <= 2:
        return 0
    elif tau <= 6:
        return 1
    elif tau <= 11:
        return 2
    elif tau <= 18:
        return 3
    else:
        return 4

tau_group_labels = ["Lag* 1–2", "Lag* 3–6", "Lag* 7–11", "Lag* 12–18", "Lag* 19–30"]
tau_colors = ["#e41a1c", "#ff7f00", "#4daf4a", "#377eb8", "#984ea3"]
forcings = ["PRCP", "SRAD", "Tair", "VP"]
season_order = ["djf", "mam", "jja", "son"]

# 52 attributes for plotting: 50 numeric + 2 categorical timing
plot_attr_list = [c for c in attr_df.columns if c not in [
    "gauge_id", "huc_02", "gauge_name", "dom_land_cover",
    "gauge_lat", "gauge_lon",
    "geol_1st_class", "geol_2nd_class",
]]

cat_attrs = ["high_prec_timing", "low_prec_timing"]

# merged_df does not yet contain the two timing columns. Re-merge to be safe.
merged_plot = wide_df.merge(attr_df, on="gauge_id", how="left")

out_pdf_path = out_pdf / "WeightedTE_vs_Attributes_52pages.pdf"

with PdfPages(out_pdf_path) as pdf:
    for attr in plot_attr_list:
        fig, axes = plt.subplots(1, 4, figsize=(20, 5))
        fig.suptitle(f"Weighted TE vs {attr}", fontsize=14, fontweight="bold")

        for ax, forcing in zip(axes, forcings):
            te_col  = f"TE_star_{forcing}"
            tau_col = f"tau_star_{forcing}"

            sub = merged_plot[[attr, te_col, tau_col]].dropna()

            # CATEGORICAL (boxplot)
            if attr in cat_attrs:
                sub = sub.copy()
                sub[attr] = sub[attr].astype(str).str.lower().str.strip()
                sub = sub[sub[attr].isin(season_order)]

                box_data = [sub[sub[attr] == s][te_col].values for s in season_order]
                ax.boxplot(box_data, positions=range(4), widths=0.4,
                           patch_artist=True,
                           boxprops=dict(facecolor="lightgrey", color="grey"),
                           medianprops=dict(color="black", linewidth=1.5),
                           whiskerprops=dict(color="grey"),
                           capprops=dict(color="grey"),
                           flierprops=dict(marker="", linestyle="none"))

                for si, season in enumerate(season_order):
                    s_sub = sub[sub[attr] == season]
                    y_vals = s_sub[te_col].values
                    taus   = s_sub[tau_col].values
                    jitter = np.random.uniform(-0.15, 0.15, size=len(y_vals))
                    for g in range(5):
                        mask = np.array([get_tau_group(t) == g for t in taus])
                        if mask.sum() > 0:
                            ax.scatter(si + jitter[mask], y_vals[mask],
                                       color=tau_colors[g], alpha=0.6,
                                       s=15, zorder=3)

                ax.set_xticks(range(4))
                ax.set_xticklabels(season_order, fontsize=8)
                ax.set_title(f"{forcing}", fontsize=10)
                ax.set_xlabel("Season", fontsize=8)

            # NUMERIC (scatter + regression)
            else:
                x = sub[attr].values.astype(float)
                y = sub[te_col].values.astype(float)
                taus = sub[tau_col].values

                valid = np.isfinite(x) & np.isfinite(y)
                x, y, taus = x[valid], y[valid], taus[valid]

                for g in range(5):
                    mask = np.array([get_tau_group(t) == g for t in taus])
                    if mask.sum() > 0:
                        ax.scatter(x[mask], y[mask],
                                   color=tau_colors[g], alpha=0.6,
                                   s=15, label=tau_group_labels[g])

                if len(x) > 2 and np.std(x) > 0:
                    slope, intercept, r, p, _ = stats.linregress(x, y)
                    x_line = np.linspace(x.min(), x.max(), 100)
                    y_line = slope * x_line + intercept
                    ax.plot(x_line, y_line, color="black",
                            linewidth=1.2, linestyle="--")
                    ax.set_title(f"{forcing}  |  r={r:.2f}  |  n={len(x)}", fontsize=10)
                else:
                    ax.set_title(f"{forcing}  |  n={len(x)}", fontsize=10)

                ax.set_xlabel(attr, fontsize=8)

            ax.set_ylabel("Weighted TE (TE*)", fontsize=8)
            ax.tick_params(labelsize=7)

        handles = [mpatches.Patch(color=tau_colors[g], label=tau_group_labels[g])
                   for g in range(5)]
        fig.legend(handles=handles, loc="lower center", ncol=5,
                   fontsize=8, title="Weighted Lag Group (τ*)",
                   title_fontsize=8, bbox_to_anchor=(0.5, -0.04))

        plt.tight_layout(rect=[0, 0.04, 1, 1])
        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)

print("Done. PDF saved to:")
print(out_pdf_path)

Done. PDF saved to:
C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Research -CAMELS\Weighted_Average_TE vs Attributes\outputs\pdf\WeightedTE_vs_Attributes_52pages.pdf


## Plot Lag* vs each attribute, colored by Lag* group

Same structure as the Peak Lag vs Attributes plot from last week. y-axis = τ* (weighted lag in days). Points are colored by τ* group using the same 5 bins. Boxplots for high_prec_timing and low_prec_timing. 52 pages total.

In [17]:
out_pdf_tau_path = out_pdf / "WeightedTau_vs_Attributes_52pages.pdf"

with PdfPages(out_pdf_tau_path) as pdf:
    for attr in plot_attr_list:
        fig, axes = plt.subplots(1, 4, figsize=(20, 5))
        fig.suptitle(f"Weighted Lag vs {attr}", fontsize=14, fontweight="bold")

        for ax, forcing in zip(axes, forcings):
            tau_col = f"tau_star_{forcing}"
            te_col  = f"TE_star_{forcing}"

            sub = merged_plot[[attr, tau_col, te_col]].dropna()

            # CATEGORICAL (boxplot)
            if attr in cat_attrs:
                sub = sub.copy()
                sub[attr] = sub[attr].astype(str).str.lower().str.strip()
                sub = sub[sub[attr].isin(season_order)]

                box_data = [sub[sub[attr] == s][tau_col].values for s in season_order]
                ax.boxplot(box_data, positions=range(4), widths=0.4,
                           patch_artist=True,
                           boxprops=dict(facecolor="lightgrey", color="grey"),
                           medianprops=dict(color="black", linewidth=1.5),
                           whiskerprops=dict(color="grey"),
                           capprops=dict(color="grey"),
                           flierprops=dict(marker="", linestyle="none"))

                for si, season in enumerate(season_order):
                    s_sub  = sub[sub[attr] == season]
                    y_vals = s_sub[tau_col].values
                    jitter = np.random.uniform(-0.15, 0.15, size=len(y_vals))
                    for g in range(5):
                        mask = np.array([get_tau_group(t) == g for t in y_vals])
                        if mask.sum() > 0:
                            ax.scatter(si + jitter[mask], y_vals[mask],
                                       color=tau_colors[g], alpha=0.6,
                                       s=15, zorder=3)

                ax.set_xticks(range(4))
                ax.set_xticklabels(season_order, fontsize=8)
                ax.set_title(f"{forcing}", fontsize=10)
                ax.set_xlabel("Season", fontsize=8)

            # NUMERIC (scatter + regression)
            else:
                x = sub[attr].values.astype(float)
                y = sub[tau_col].values.astype(float)

                valid = np.isfinite(x) & np.isfinite(y)
                x, y = x[valid], y[valid]

                # color by τ* group (same as coloring by y value)
                for g in range(5):
                    mask = np.array([get_tau_group(t) == g for t in y])
                    if mask.sum() > 0:
                        ax.scatter(x[mask], y[mask],
                                   color=tau_colors[g], alpha=0.6,
                                   s=15, label=tau_group_labels[g])

                if len(x) > 2 and np.std(x) > 0:
                    slope, intercept, r, p, _ = stats.linregress(x, y)
                    x_line = np.linspace(x.min(), x.max(), 100)
                    y_line = slope * x_line + intercept
                    ax.plot(x_line, y_line, color="black",
                            linewidth=1.2, linestyle="--")
                    ax.set_title(f"{forcing}  |  r={r:.2f}  |  n={len(x)}", fontsize=10)
                else:
                    ax.set_title(f"{forcing}  |  n={len(x)}", fontsize=10)

                ax.set_xlabel(attr, fontsize=8)

            ax.set_ylabel("Weighted Lag τ* (days)", fontsize=8)
            ax.tick_params(labelsize=7)

        # shared legend
        tau_group_labels_lag = ["Lag* 1–2", "Lag* 3–6", "Lag* 7–11", "Lag* 12–18", "Lag* 19–30"]
        handles = [mpatches.Patch(color=tau_colors[g], label=tau_group_labels_lag[g])
                   for g in range(5)]
        fig.legend(handles=handles, loc="lower center", ncol=5,
                   fontsize=8, title="Weighted Lag Group",
                   title_fontsize=8, bbox_to_anchor=(0.5, -0.04))

        plt.tight_layout(rect=[0, 0.04, 1, 1])
        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)

print("Done. PDF saved to:")
print(out_pdf_tau_path)

Done. PDF saved to:
C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Research -CAMELS\Weighted_Average_TE vs Attributes\outputs\pdf\WeightedTau_vs_Attributes_52pages.pdf


## CONUS maps — discrete color bins for τ* and TE*

- τ* map: 5 discrete lag bins, same colors as scatter plots
- TE* map: 5 discrete percentile bins per forcing (20/40/60/80th percentile)
- CONUS only, tight extent

In [31]:
out_map_path = out_pdf / "CONUS_maps_weighted_tau_TE.pdf"
proj = ccrs.LambertConformal(central_longitude=-96, central_latitude=37.5)

# τ* bins
tau_bin_edges  = [0, 2, 6, 11, 18, 30]
tau_bin_labels = ["Lag* 1–2", "Lag* 3–6", "Lag* 7–11", "Lag* 12–18", "Lag* 19–30"]
tau_bin_colors = ["#e41a1c", "#ff7f00", "#4daf4a", "#377eb8", "#984ea3"]

def get_tau_bin(val):
    for i in range(len(tau_bin_edges) - 1):
        if tau_bin_edges[i] < val <= tau_bin_edges[i + 1]:
            return i
    return len(tau_bin_labels) - 1

# TE* bins
te_bin_labels = ["Very Low", "Low", "Medium", "High", "Very High"]
te_bin_colors = ["#4a0080", "#66c266", "#ffd700", "#ff8c00", "#8b0000"]

def get_te_bin(val, edges):
    for i in range(len(edges) - 1):
        if edges[i] <= val <= edges[i + 1]:
            return i
    return len(te_bin_labels) - 1

def draw_map_base(ax):
    ax.set_extent([-122, -68, 23, 50], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND, facecolor="whitesmoke")
    ax.add_feature(cfeature.OCEAN, facecolor="#d4eaf7")
    ax.add_feature(cfeature.LAKES, facecolor="#d4eaf7")
    ax.add_feature(cfeature.COASTLINE, linewidth=0.6)
    ax.add_feature(cfeature.BORDERS, linewidth=0.9, edgecolor="black")
    states = cfeature.NaturalEarthFeature(
        category="cultural",
        name="admin_1_states_provinces_lines",
        scale="50m", facecolor="none")
    ax.add_feature(states, edgecolor="grey", linewidth=0.5)

def add_state_names(ax):
    shpfile = natural_earth(resolution="50m", category="cultural",
                            name="admin_1_states_provinces")
    reader = Reader(shpfile)
    for record in reader.records():
        if record.attributes["admin"] != "United States of America":
            continue
        name = record.attributes["name"]
        geom = record.geometry
        cx = geom.centroid.x
        cy = geom.centroid.y
        if cx < -122 or cx > -68 or cy < 23 or cy > 50:
            continue
        ax.text(cx, cy, name, fontsize=5, ha="center", va="center",
                color="dimgrey", transform=ccrs.PlateCarree(),
                fontweight="bold", zorder=4)

def add_title_box(ax, text):
    ax.text(0.02, 0.97, text, transform=ax.transAxes,
            fontsize=11, fontweight="bold", va="top", ha="left",
            bbox=dict(facecolor="white", edgecolor="grey",
                      boxstyle="round,pad=0.3", alpha=0.85),
            zorder=10)

def add_vertical_legend(ax, colors, labels, title):
    handles = [mpatches.Patch(color=colors[g], label=labels[g])
               for g in range(len(labels))]
    ax.legend(handles=handles,
              loc="upper left",
              bbox_to_anchor=(1.01, 1.0),
              borderaxespad=0,
              fontsize=11,
              title=title,
              title_fontsize=12,
              framealpha=0.9,
              edgecolor="grey",
              handlelength=2.0,
              handleheight=2.0)

with PdfPages(out_map_path) as pdf:

    # Page 1: τ*
    fig, axes = plt.subplots(2, 2, figsize=(24, 15),
                             subplot_kw={"projection": proj})
    fig.suptitle("Weighted Lag τ* by Forcing — 671 CAMELS Basins",
                 fontsize=15, fontweight="bold", y=1.01)
    axes = axes.flatten()

    for ax, forcing in zip(axes, forcings):
        tau_col = f"tau_star_{forcing}"
        sub = map_df[["gauge_lat", "gauge_lon", tau_col]].dropna()

        draw_map_base(ax)
        add_state_names(ax)

        for g in range(len(tau_bin_labels)):
            mask = sub[tau_col].apply(get_tau_bin) == g
            s = sub[mask]
            if len(s) > 0:
                ax.scatter(s["gauge_lon"], s["gauge_lat"],
                           color=tau_bin_colors[g],
                           s=20, alpha=0.9, zorder=5,
                           transform=ccrs.PlateCarree())

        add_title_box(ax, f"{forcing}  |  n = {len(sub)}")
        add_vertical_legend(ax, tau_bin_colors, tau_bin_labels, "τ* group")

    plt.tight_layout(rect=[0, 0, 1, 0.98])
    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)

    # Page 2: TE*
    fig, axes = plt.subplots(2, 2, figsize=(24, 15),
                             subplot_kw={"projection": proj})
    fig.suptitle("Weighted TE (TE*) by Forcing — 671 CAMELS Basins",
                 fontsize=15, fontweight="bold", y=1.01)
    axes = axes.flatten()

    for ax, forcing in zip(axes, forcings):
        te_col = f"TE_star_{forcing}"
        sub = map_df[["gauge_lat", "gauge_lon", te_col]].dropna()

        vals = sub[te_col].values
        edges = [vals.min(),
                 np.percentile(vals, 20),
                 np.percentile(vals, 40),
                 np.percentile(vals, 60),
                 np.percentile(vals, 80),
                 vals.max()]

        draw_map_base(ax)
        add_state_names(ax)

        for g in range(len(te_bin_labels)):
            mask = sub[te_col].apply(lambda v: get_te_bin(v, edges)) == g
            s = sub[mask]
            if len(s) > 0:
                ax.scatter(s["gauge_lon"], s["gauge_lat"],
                           color=te_bin_colors[g],
                           s=20, alpha=0.9, zorder=5,
                           transform=ccrs.PlateCarree())

        add_title_box(ax, f"{forcing}  |  n = {len(sub)}")
        pct_ranges = ["0–20th", "20–40th", "40–60th", "60–80th", "80–100th"]
        te_labels_with_range = [
            f"{te_bin_labels[g]}\n({pct_ranges[g]} percentiles)\n({edges[g]:.4f} – {edges[g+1]:.4f})"
            for g in range(len(te_bin_labels))
        ]
        add_vertical_legend(ax, te_bin_colors, te_labels_with_range, "TE* group")

    plt.tight_layout(rect=[0, 0, 1, 0.98])
    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)

print("Done. Maps saved to:")
print(out_map_path)

Done. Maps saved to:
C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Research -CAMELS\Weighted_Average_TE vs Attributes\outputs\pdf\CONUS_maps_weighted_tau_TE.pdf


## CONUS Map Interpretation

### Page 1 — Weighted Lag τ* Map (in days)

**PRCP:**
- Green (Lag* 7–11) is the most common color across most of CONUS,
  especially across the Midwest, Central U.S., and parts of the South.
- Orange (Lag* 3–6) is clearly visible across many southern basins,
  including Texas, Louisiana, Mississippi, Alabama, Georgia, and
  Florida. Some orange also appears in the Midwest.
- Blue (Lag* 12–18) appears in the Pacific Northwest (Washington,
  Oregon) and parts of the Northeast.
- Red (Lag* 1–2) and purple (Lag* 19–30) are very rare and scattered.
- Overall, PRCP shows the shortest weighted lags among all 4 forcings.

**SRAD:**
- Blue (Lag* 12–18) is the most common color and is widespread across
  the West, Midwest, and Northeast.
- Green (Lag* 7–11) is also common but less than blue.
- Purple (Lag* 19–30) appears more often than in PRCP, especially in
  the central and eastern parts of CONUS.
- Orange and red are rare.
- Overall, SRAD shows longer weighted lags than PRCP.

**Tair:**
- Blue (Lag* 12–18) dominates the western half of CONUS, including
  the Pacific Northwest, Rockies, and much of the West.
- Purple (Lag* 19–30) is very common in the eastern half of CONUS,
  including the Midwest, Northeast, and parts of the Southeast.
- Green appears in much fewer basins than in PRCP and SRAD.
- Red and orange are very rare.
- Overall, Tair shows the longest weighted lags among all 4 forcings.

**VP:**
- Blue (Lag* 12–18) is the most common color across the map.
- Purple (Lag* 19–30) is also common, especially in the eastern half
  of CONUS.
- Green appears in many basins too.
- Orange and red are very rare.
- Overall, VP shows lag lengths between SRAD and Tair.

**Overall τ* pattern across forcings:**
The weighted lag clearly increases in the order PRCP → SRAD → VP → Tair.
The color shift is direct evidence — more orange and green in PRCP,
more blue and purple in Tair. Mean values from previous steps confirm:
PRCP ≈ 8.6 days, SRAD ≈ 12.4 days, VP ≈ 14.0 days, Tair ≈ 15.4 days.
This ordering is physically interpretable. PRCP drives runoff quickly,
while Tair and VP affect streamflow through slower processes like
snowmelt and seasonal ET.

---

### Page 2 — Weighted TE* Map

**Important:** Each forcing has its own percentile-based color bins.
Colors cannot be compared across the 4 subplots.

**PRCP:**
- Very High TE* (dark red) is clearly clustered along the Pacific
  Northwest coast (Washington, Oregon, northern California) and parts
  of the Northeast coast.
- Very Low TE* (dark purple) appears in scattered basins across the
  map, including parts of Texas, the Mid-Atlantic, and the arid West.
- Yellow and orange (Medium and High) are widespread across the
  Midwest and Southeast.

**SRAD:**
- Very High TE* (dark red) is most clearly concentrated along the
  Pacific Northwest coast and in parts of the Rocky Mountain region.
- Very Low TE* (dark purple) appears scattered across many regions,
  including parts of Texas, the Carolinas area, and the Great Plains.
- Outside the Pacific Northwest cluster, the colors are quite mixed
  across the rest of CONUS.

**Tair:**
- Very High TE* (dark red) is visible along the Pacific Northwest coast
  and scattered in other regions.
- Very Low TE* (dark purple) is widely scattered, including more in the
  central east, texas area.
- The overall pattern is more mixed compared to PRCP and SRAD.

**VP:**
- Very High TE* (dark red) is again visible along the Pacific Northwest
  coast and parts of the Northeast.
- Very Low TE* (dark purple) is scattered across many regions.
- The pattern is broadly similar to Tair.

**Overall TE* pattern:**
For all 4 forcings, the Very High TE* basins are most clearly clustered
along the Pacific Northwest coast. This is the only strong, consistent
geographic pattern visible across all 4 maps. The rest of CONUS shows
mixed colors, so no other clean geographic claim can be made.

---

### Caution
- TE* colors are percentile-based per forcing. Do not compare colors
  across the 4 subplots to rank which forcing is stronger overall.
- τ* is a weighted centroid. Basins with TE spread across many lags
  end up with mid-range τ* by arithmetic, not necessarily from a
  single physical mechanism.
- High TE* indicates strong information transfer, not direct causation.

## Top 10 attributes by |r| for τ* and TE*

For each forcing, rank the 50 attributes by absolute Pearson r and
print the top 10. We do this twice — once for τ* (weighted lag) and 
once for TE* (weighted TE).

In [32]:
# add absolute r columns
corr_df["abs_r_tau"] = corr_df["r_tau"].abs()
corr_df["abs_r_TE"]  = corr_df["r_TE"].abs()

forcings = ["PRCP", "SRAD", "Tair", "VP"]

# Top 10 by |r| for TE*
print("=" * 65)
print("TOP 10 ATTRIBUTES BY |r| FOR WEIGHTED TE (TE*)")
print("=" * 65)
for f in forcings:
    sub = corr_df[corr_df["forcing"] == f].sort_values(
        "abs_r_TE", ascending=False).head(10)
    n = int(sub["n_TE"].max())
    print(f"\n--- {f} (n={n}) ---")
    for i, row in enumerate(sub.itertuples(), 1):
        print(f"  {i:2}. {row.attribute:<30} r = {row.r_TE:+.3f}")

# Top 10 by |r| for τ*
print("\n")
print("=" * 65)
print("TOP 10 ATTRIBUTES BY |r| FOR WEIGHTED LAG (τ*)")
print("=" * 65)
for f in forcings:
    sub = corr_df[corr_df["forcing"] == f].sort_values(
        "abs_r_tau", ascending=False).head(10)
    n = int(sub["n_tau"].max())
    print(f"\n--- {f} (n={n}) ---")
    for i, row in enumerate(sub.itertuples(), 1):
        print(f"  {i:2}. {row.attribute:<30} r = {row.r_tau:+.3f}")

print("\nDone.")

TOP 10 ATTRIBUTES BY |r| FOR WEIGHTED TE (TE*)

--- PRCP (n=670) ---
   1. q95                            r = +0.440
   2. runoff_ratio                   r = +0.399
   3. q_mean                         r = +0.388
   4. low_prec_freq                  r = -0.380
   5. high_prec_freq                 r = -0.363
   6. p_mean                         r = +0.316
   7. p_seasonality                  r = -0.289
   8. aridity                        r = -0.250
   9. q5                             r = +0.248
  10. slope_mean                     r = +0.235

--- SRAD (n=653) ---
   1. q95                            r = +0.606
   2. runoff_ratio                   r = +0.561
   3. q_mean                         r = +0.533
   4. high_prec_freq                 r = -0.458
   5. slope_mean                     r = +0.431
   6. p_seasonality                  r = -0.426
   7. low_prec_freq                  r = -0.414
   8. p_mean                         r = +0.379
   9. q5                             r = +0.2

## Rank attributes by |r|, save CSV and PDF, show top 10

For each forcing, rank all 50 attributes by absolute Pearson r — once
for τ* and once for TE*. Save full rankings as CSV and PDF. Print top 10 in the notebook for quick reference.

In [34]:
# add absolute r columns
corr_df["abs_r_tau"] = corr_df["r_tau"].abs()
corr_df["abs_r_TE"]  = corr_df["r_TE"].abs()

forcings = ["PRCP", "SRAD", "Tair", "VP"]

# Build ranked tables
ranked_TE = []
ranked_tau = []
for f in forcings:
    sub_TE = corr_df[corr_df["forcing"] == f].sort_values(
        "abs_r_TE", ascending=False).reset_index(drop=True)
    sub_TE["rank"] = sub_TE.index + 1
    ranked_TE.append(sub_TE[["forcing", "rank", "attribute", "n_TE", "r_TE", "abs_r_TE"]])

    sub_tau = corr_df[corr_df["forcing"] == f].sort_values(
        "abs_r_tau", ascending=False).reset_index(drop=True)
    sub_tau["rank"] = sub_tau.index + 1
    ranked_tau.append(sub_tau[["forcing", "rank", "attribute", "n_tau", "r_tau", "abs_r_tau"]])

ranked_TE_df  = pd.concat(ranked_TE, ignore_index=True)
ranked_tau_df = pd.concat(ranked_tau, ignore_index=True)

# Save CSVs
ranked_TE_path  = out_csv / "ranked_attributes_TE_star.csv"
ranked_tau_path = out_csv / "ranked_attributes_tau_star.csv"
ranked_TE_df.to_csv(ranked_TE_path,  index=False)
ranked_tau_df.to_csv(ranked_tau_path, index=False)

print("Saved CSV:", ranked_TE_path)
print("Saved CSV:", ranked_tau_path)

# Save PDF with ranking tables
ranked_pdf_path = out_pdf / "Ranked_attributes_weighted_TE_tau.pdf"

def build_ranking_page(pdf, title, sub_df, r_col, n_col):
    n = int(sub_df[n_col].max())

    fig = plt.figure(figsize=(8.5, 13))
    fig.suptitle(f"{title}\n{sub_df['forcing'].iloc[0]}  |  n = {n}",
                 fontsize=14, fontweight="bold", y=0.97)

    ax = fig.add_axes([0.05, 0.02, 0.90, 0.88])
    ax.axis("off")

    table_data = [
        [int(row["rank"]), row["attribute"], f"{row[r_col]:+.3f}"]
        for _, row in sub_df.iterrows()
    ]
    table = ax.table(
        cellText=table_data,
        colLabels=["Rank", "Attribute", "r"],
        loc="upper center",
        cellLoc="center",
        colWidths=[0.12, 0.55, 0.15],
    )
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1, 1.35)

    # header style
    for j in range(3):
        cell = table[(0, j)]
        cell.set_facecolor("#4a90d9")
        cell.set_text_props(color="white", fontweight="bold")
        cell.set_height(0.022)

    n_rows = len(table_data)
    for i in range(1, n_rows + 1):
        for j in range(3):
            cell = table[(i, j)]
            cell.set_height(0.020)
            if i <= 10:
                cell.set_facecolor("#fff2cc")
            else:
                cell.set_facecolor("#f7f7f7" if i % 2 == 0 else "white")
            if j == 1:
                cell.set_text_props(ha="left")
            cell.set_edgecolor("grey")

    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)

with PdfPages(ranked_pdf_path) as pdf:
    # TE* pages
    for f in forcings:
        sub = ranked_TE_df[ranked_TE_df["forcing"] == f].copy()
        build_ranking_page(pdf, "Ranked Attributes by |r| — Weighted TE (TE*)",
                           sub, "r_TE", "n_TE")

    # τ* pages
    for f in forcings:
        sub = ranked_tau_df[ranked_tau_df["forcing"] == f].copy()
        build_ranking_page(pdf, "Ranked Attributes by |r| — Weighted Lag (τ*)",
                           sub, "r_tau", "n_tau")

print("Saved PDF:", ranked_pdf_path)

# Print top 10 in notebook
print("\n")
print("=" * 65)
print("TOP 10 ATTRIBUTES BY |r| FOR WEIGHTED TE (TE*)")
print("=" * 65)
for f in forcings:
    sub = ranked_TE_df[ranked_TE_df["forcing"] == f].head(10)
    n = int(sub["n_TE"].max())
    print(f"\n--- {f} (n={n}) ---")
    for _, row in sub.iterrows():
        print(f"  {int(row['rank']):2}. {row['attribute']:<30} r = {row['r_TE']:+.3f}")

print("\n")
print("=" * 65)
print("TOP 10 ATTRIBUTES BY |r| FOR WEIGHTED LAG (τ*)")
print("=" * 65)
for f in forcings:
    sub = ranked_tau_df[ranked_tau_df["forcing"] == f].head(10)
    n = int(sub["n_tau"].max())
    print(f"\n--- {f} (n={n}) ---")
    for _, row in sub.iterrows():
        print(f"  {int(row['rank']):2}. {row['attribute']:<30} r = {row['r_tau']:+.3f}")

print("\nDone.")

Saved CSV: C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Research -CAMELS\Weighted_Average_TE vs Attributes\outputs\summary_csv\ranked_attributes_TE_star.csv
Saved CSV: C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Research -CAMELS\Weighted_Average_TE vs Attributes\outputs\summary_csv\ranked_attributes_tau_star.csv
Saved PDF: C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Research -CAMELS\Weighted_Average_TE vs Attributes\outputs\pdf\Ranked_attributes_weighted_TE_tau.pdf


TOP 10 ATTRIBUTES BY |r| FOR WEIGHTED TE (TE*)

--- PRCP (n=670) ---
   1. q95                            r = +0.440
   2. runoff_ratio                   r = +0.399
   3. q_mean                         r = +0.388
   4. low_prec_freq                  r = -0.380
   5. high_prec_freq                 r = -0.363
   6. p_mean                         r = +0.316
   7. p_seasonality                  r = -0.289
   8. aridity                        r = -0.25

## Side-by-side comparison PDF — Peak vs Weighted

Compare last week's Peak TE / Peak Lag rankings with this week's
Weighted TE* / Weighted τ* rankings. 8 pages — 4 forcings × 2 metrics
(TE and Lag). Each side independently ranked descending by |r|. Top 10
on each side highlighted yellow. Thick black line in the middle.

In [35]:
# load last week's peak correlation file
peak_path = r"C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Python\Jupyter Notebook\CAMELS Research\Final research code\TE basin to basin analysis with 60 attributes\outputs\summary_csv\correlation_summary_TE_lag_vs_attributes.csv"

peak_df = pd.read_csv(peak_path)
peak_df["abs_r_peak_TE"]  = peak_df["r_peak_TE"].abs()
peak_df["abs_r_peak_lag"] = peak_df["r_peak_lag"].abs()

print("Peak df shape:", peak_df.shape)
print("Peak df forcings:", peak_df["forcing"].unique())

forcings = ["PRCP", "SRAD", "Tair", "VP"]
compare_pdf_path = out_pdf / "Compare_Peak_vs_Weighted_attributes.pdf"


def build_compare_page(pdf, forcing, metric_name, peak_sub, weighted_sub,
                       peak_r_col, peak_n_col, w_r_col, w_n_col):
    n_peak = int(peak_sub[peak_n_col].max())
    n_w    = int(weighted_sub[w_n_col].max())

    fig = plt.figure(figsize=(14, 13))
    fig.suptitle(
        f"Comparison — {metric_name}\n{forcing}  |  Peak n = {n_peak}   |   Weighted n = {n_w}",
        fontsize=14, fontweight="bold", y=0.97
    )

    # left table axes
    ax_left = fig.add_axes([0.02, 0.02, 0.46, 0.88])
    ax_left.axis("off")
    ax_left.set_title(f"Peak {metric_name}", fontsize=12,
                      fontweight="bold", pad=8, color="#333333")

    left_data = [
        [int(row["rank"]), row["attribute"], f"{row[peak_r_col]:+.3f}"]
        for _, row in peak_sub.iterrows()
    ]
    left_table = ax_left.table(
        cellText=left_data,
        colLabels=["Rank", "Attribute", "r"],
        loc="upper center",
        cellLoc="center",
        colWidths=[0.12, 0.60, 0.18],
    )
    left_table.auto_set_font_size(False)
    left_table.set_fontsize(8)
    left_table.scale(1, 1.25)

    # right table axes
    ax_right = fig.add_axes([0.52, 0.02, 0.46, 0.88])
    ax_right.axis("off")
    ax_right.set_title(f"Weighted {metric_name}", fontsize=12,
                       fontweight="bold", pad=8, color="#333333")

    right_data = [
        [int(row["rank"]), row["attribute"], f"{row[w_r_col]:+.3f}"]
        for _, row in weighted_sub.iterrows()
    ]
    right_table = ax_right.table(
        cellText=right_data,
        colLabels=["Rank", "Attribute", "r"],
        loc="upper center",
        cellLoc="center",
        colWidths=[0.12, 0.60, 0.18],
    )
    right_table.auto_set_font_size(False)
    right_table.set_fontsize(8)
    right_table.scale(1, 1.25)

    # style both tables
    for table, data in [(left_table, left_data), (right_table, right_data)]:
        for j in range(3):
            cell = table[(0, j)]
            cell.set_facecolor("#4a90d9")
            cell.set_text_props(color="white", fontweight="bold")
            cell.set_height(0.022)

        for i in range(1, len(data) + 1):
            for j in range(3):
                cell = table[(i, j)]
                cell.set_height(0.018)
                if i <= 10:
                    cell.set_facecolor("#fff2cc")
                else:
                    cell.set_facecolor("#f7f7f7" if i % 2 == 0 else "white")
                if j == 1:
                    cell.set_text_props(ha="left")
                cell.set_edgecolor("grey")

    # thick black divider between left and right tables
    fig.add_artist(plt.Line2D([0.50, 0.50], [0.02, 0.93],
                               color="black", linewidth=2.5))

    pdf.savefig(fig, bbox_inches="tight")
    plt.close(fig)


with PdfPages(compare_pdf_path) as pdf:
    for f in forcings:
        # TE comparison
        peak_TE  = peak_df[peak_df["forcing"] == f].sort_values(
            "abs_r_peak_TE", ascending=False).reset_index(drop=True)
        peak_TE["rank"] = peak_TE.index + 1

        w_TE = ranked_TE_df[ranked_TE_df["forcing"] == f].reset_index(drop=True)

        build_compare_page(pdf, f, "TE",
                           peak_TE, w_TE,
                           "r_peak_TE", "n_peak_TE",
                           "r_TE", "n_TE")

        # Lag comparison
        peak_lag = peak_df[peak_df["forcing"] == f].sort_values(
            "abs_r_peak_lag", ascending=False).reset_index(drop=True)
        peak_lag["rank"] = peak_lag.index + 1

        w_tau = ranked_tau_df[ranked_tau_df["forcing"] == f].reset_index(drop=True)

        build_compare_page(pdf, f, "Lag",
                           peak_lag, w_tau,
                           "r_peak_lag", "n_peak_lag",
                           "r_tau", "n_tau")

print("Saved comparison PDF:", compare_pdf_path)

Peak df shape: (200, 8)
Peak df forcings: ['PRCP' 'SRAD' 'Tair' 'VP']
Saved comparison PDF: C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Research -CAMELS\Weighted_Average_TE vs Attributes\outputs\pdf\Compare_Peak_vs_Weighted_attributes.pdf


## Comparison of Peak vs Weighted — Key Observations

### Similarities (what stayed the same)

**1. The top attributes are mostly the same — just reshuffled.**
For every forcing, the top 10 attributes from Peak TE and Weighted TE
overlap heavily. The same 7–8 attributes appear on both sides for each
forcing. Examples:
- PRCP TE: q95, q_mean, runoff_ratio, low_prec_freq, high_prec_freq,
  p_mean, p_seasonality appear in both top 10 lists.
- SRAD TE: q95, runoff_ratio, q_mean, slope_mean, p_seasonality,
  high_prec_freq, low_prec_freq, p_mean, q5 all in both top 10.
- Tair TE: runoff_ratio, high_prec_freq, q95, low_prec_freq, q_mean,
  stream_elas, p_seasonality, slope_mean — same dominant set.
- VP TE: runoff_ratio, high_prec_freq, low_prec_freq, q95, q_mean,
  stream_elas — same dominant attributes.

**2. Sign of correlation stays the same.**
Every attribute keeps its +/- sign moving from Peak to Weighted. So
the physical direction of the relationship is preserved.

**3. The strongest attribute for each forcing is consistent.**
- PRCP TE: q95 stays #1 in both (Peak r = +0.624, Weighted r = +0.440)
- SRAD TE: q95 stays #1 in both (Peak r = +0.603, Weighted r = +0.606)
- Tair TE: runoff_ratio and high_prec_freq dominate both
- VP TE: runoff_ratio and high_prec_freq dominate both

---

### Differences (what changed)

**1. Magnitude of |r| generally drops slightly for Weighted TE.**
Most attributes show slightly weaker correlations in the weighted case.
For example PRCP TE:
- q95: 0.624 → 0.440
- q_mean: 0.612 → 0.388
- p_mean: 0.588 → 0.316


**2. Lag side (τ*) shows the biggest changes.**

For Peak Lag vs Weighted Lag, the top 10 attributes shift more than
on the TE side:

- **PRCP Lag:** Peak top 10 was elev_mean, frac_snow, aridity,
  hfd_mean, p_mean, root_depth_50, gvf_max, lai_max, high_q_dur,
  slope_fdc. Weighted top 10 is elev_mean, root_depth_50, slope_mean,
  frac_snow, lai_diff, gvf_diff, high_prec_dur, gvf_max,
  soil_depth_pelletier, aridity. Six attributes overlap, four
  attributes are different. elev_mean stays #1 but with a stronger
  r (+0.326 → +0.417).

- **SRAD Lag:** The biggest reshuffle. Peak top 10 had p_seasonality,
  gvf_diff, silt_frac, q95, slope_mean, p_mean, q_mean, runoff_ratio,
  q5, pet_mean. Weighted top 10 is sand_frac, silt_frac, slope_mean,
  pet_mean, elev_mean, soil_conductivity, frac_snow, geol_permeability,
  soil_porosity, soil_depth_pelletier. Soil and topography attributes
  rise to the top for Weighted, while climate attributes (p_seasonality,
  q95, q_mean) drop out. This is a notable shift.

- **Tair Lag:** Peak had frac_snow (-0.407), slope_mean, elev_mean,
  high_prec_freq, runoff_ratio, hfd_mean, glim_1st_class_frac. Weighted
  has root_depth_50, hfd_mean, slope_mean, lai_diff, elev_mean, lai_max,
  frac_snow. Vegetation attributes (root_depth_50, lai_max, lai_diff)
  become more important in the weighted case.

- **VP Lag:** Peak had frac_snow, elev_mean, lai_diff, high_prec_dur,
  hfd_mean, slope_mean. Weighted top 10 leads with lai_diff, lai_max,
  high_prec_dur, low_prec_dur, gvf_max, root_depth_50. Vegetation and
  precipitation timing attributes dominate the weighted case.

**3. Lag-side correlations are weaker overall in Weighted.**
For all 4 forcings, even the #1 attribute in Weighted Lag has
|r| ≤ 0.45 in most cases, while Peak Lag had several |r| > 0.40. The
weighted lag is a centroid, which smooths out the sharp peaks that
correlated more strongly with attributes.

---

### Key Patterns

**1. TE side is robust — Peak and Weighted tell the same story.**
The dominant attributes for TE strength are climate and streamflow
metrics (q95, runoff_ratio, q_mean, high_prec_freq, low_prec_freq,
p_mean, p_seasonality, stream_elas). These dominate both Peak TE and
Weighted TE for every forcing. So switching from peak to weighted
does not change which attributes matter for TE magnitude — only the
exact strength.

**2. Lag side is sensitive — Peak and Weighted tell different stories.**
The dominant attributes shift between Peak Lag and Weighted Lag,
especially for SRAD, Tair, and VP. In the weighted case, soil,
topography, and vegetation attributes become more prominent
(soil_depth, root_depth, lai_max, lai_diff, gvf_max, elev_mean,
slope_mean). In the peak case, climate timing attributes
(p_seasonality, frac_snow, high_prec_freq) were more prominent.


**3. Caution.**
Even after this analysis, no attribute reaches |r| > 0.65 anywhere.
Most top correlations sit in the 0.3–0.5 range. This means basin
attributes explain part of the variation in TE and lag, but a large
portion remains unexplained. The weighted approach does not solve
the scattered-relationship problem from last week — it gives a
different angle on the same data.